In [2]:
import pandas as pd
import numpy as np

In [4]:
sail_cluster_data = pd.read_excel( "Sail_Cluster.xlsx")

In [6]:
sail_cluster_data.columns

Index(['speed', 'mean_draft', 'sea_state', 'me_actual_steaming_time',
       'ae_t_steaming', 'aux_running', 'blr_running', 'Cluster'],
      dtype='object')

In [8]:
sail_cluster_data.head()

,speed,mean_draft,sea_state,me_actual_steaming_time,ae_t_steaming,aux_running,blr_running,Cluster
0,14.39,5.95,5,20.5,21.3,3,0,1
1,7.50,5.95,4,0.8,6.5,3,1,1
2,14.22,5.95,2,19.9,20.8,2,0,1
3,4.62,5.95,2,1.3,7.4,3,1,1
4,8.44,5.95,2,9.0,27.0,3,1,1


In [10]:
sail_cluster_data.shape

(786, 8)

In [12]:
sail_cluster_data['me_actual_steaming_time'].value_counts()

me_actual_steaming_time
24.00    76
1.00     26
1.30     25
2.00     23
3.00     17
         ..
19.40     1
5.12      1
6.24      1
5.80      1
15.40     1
Name: count, Length: 196, dtype: int64

In [14]:
sail_cluster_data.shape  

(786, 8)

# Sailing Data Profiles

In [16]:
def generate_sail_profiles(cluster_data, 
                      no_of_profiles,
                      user_draft_range=2, 
                      user_speed_range=4,
                      user_sea_state_range=2,
                      user_me_time_range=4,  
                      user_ae_time_range=4    
                     ):
    
    global finaldict
    finaldict = {}
    
    for index in range(1, no_of_profiles + 1):
      
        # Filter data for the cluster
        data = cluster_data[cluster_data['Cluster'] == index][['speed', 'mean_draft', 'me_actual_steaming_time', 'sea_state',
                                                               'ae_t_steaming', 'aux_running', 'blr_running']].copy()

        # Convert columns to numeric and round them
        data['speed'] = pd.to_numeric(data['speed'], errors='coerce').round(0)
        data['mean_draft'] = pd.to_numeric(data['mean_draft'], errors='coerce').round(0)

        # Create a pivot table for speed and draft data
        melt = pd.melt(data, id_vars=['mean_draft'], value_vars=['speed'])
        melt['speed'] = melt['value']
        pd2 = (melt.pivot_table(index='speed', columns='mean_draft', values='value', aggfunc=len, fill_value=0)
               .reset_index()
               .rename_axis(None, axis=1))
        pd2 = pd2.set_index('speed')
        pd2.index.name = None
        total = np.sum(pd2.to_numpy())
        pd2 = round((pd2 * 100 / total), 2)
        pd3 = pd2.reset_index()
        print(pd3.columns)


        # Rename the column
        pd3.rename(columns={'index': 'Speed/Draft'}, inplace=True)
        pd3['Draft'] = pd.Series(pd3.columns[1:])

        # Get draft and speed percentages
        mean_draft_percent_df = (data['mean_draft'].round(0).value_counts(normalize=True) * 100).round(1).sort_index()
        pd3['Draft_%'] = pd3['Draft'].map(dict(zip(mean_draft_percent_df.index, mean_draft_percent_df)))

        speed_percent_df = (data['speed'].round(0).value_counts(normalize=True) * 100).round(1).sort_index()
        pd3['Speed'] = pd3['Speed/Draft']
        pd3['Speed_%'] = pd3['Speed/Draft'].map(dict(zip(speed_percent_df.index, speed_percent_df)))

        # Extend the DataFrame to match the desired length (30 rows)
        pd3 = pd3.reindex(range(30))

        # Create percentage columns for Sea State, ME steaming time, AE steaming time, AE running, and BLR running
        ss_df = (data['sea_state'].round(0).value_counts(normalize=True) * 100).round(1).sort_index()
        pd3['Sea_State'] = pd.Series(sorted(abs(data['sea_state'] - 10).round().value_counts().index))
        pd3['Sea_State%'] = pd3['Sea_State'].map(dict(zip(abs(ss_df.index - 10)[::-1], ss_df)))

        me_steam_time_df = (data['me_actual_steaming_time'].round(0).value_counts(normalize=True) * 100).round(1).sort_index()
        pd3['ME_Time'] = pd.Series(sorted(data['me_actual_steaming_time'].round().value_counts().index))
        pd3['ME_Time%'] = pd3['ME_Time'].map(dict(zip(me_steam_time_df.index, me_steam_time_df)))

        ae_steam_time_df = (data['ae_t_steaming'].round(0).value_counts(normalize=True) * 100).round(1).sort_index()
        pd3['AE_Time'] = pd.Series(sorted(data['ae_t_steaming'].round().value_counts().index))
        pd3['AE_Time%'] = pd3['AE_Time'].map(dict(zip(ae_steam_time_df.index, ae_steam_time_df)))

        ae_running_df = (data['aux_running'].round(0).value_counts(normalize=True) * 100).round(1).sort_index()
        pd3['AE_Running'] = pd.Series(sorted(data['aux_running'].round().value_counts().index))
        pd3['AE_Running%'] = pd3['AE_Running'].map(dict(zip(ae_running_df.index, ae_running_df)))

        blr_running_df = (data['blr_running'].round(0).value_counts(normalize=True) * 100).round(1).sort_index()
        pd3['BLR_Running'] = pd.Series(sorted(data['blr_running'].round().value_counts().index))
        pd3['BLR_Running%'] = pd3['BLR_Running'].map(dict(zip(blr_running_df.index, blr_running_df)))

        # Create range DataFrame
        range_df = pd.DataFrame(index=range(10))

        # Process ranges for draft, speed, sea state, ME time, AE time, AE running, and BLR running
        data_profile = pd3.iloc[:, -14:].copy()

        data_profile['draft_range'] = pd.cut(data_profile['Draft'], bins=list(range(0, int(data_profile['Draft'].max()) + user_draft_range, user_draft_range))).astype(str)
        range_profile_draft = data_profile.groupby('draft_range')['Draft_%'].sum().reset_index()
        range_df['Draft_Range'] = range_profile_draft['draft_range']
        range_df['Draft_%'] = range_profile_draft['Draft_%']

        data_profile['speed_range'] = pd.cut(data_profile['Speed'], bins=list(range(0, int(data_profile['Speed'].max()) + user_speed_range, user_speed_range))).astype(str)
        range_profile_speed = data_profile.groupby('speed_range')['Speed_%'].sum().reset_index()
        range_df['Speed_Range'] = range_profile_speed['speed_range']
        range_df['Speed_%'] = range_profile_speed['Speed_%']

        data_profile['sea_state_range'] = pd.cut(data_profile['Sea_State'], bins=list(range(0, int(data_profile['Sea_State'].max()) + user_sea_state_range, user_sea_state_range))).astype(str)
        range_profile_sea_state = data_profile.groupby('sea_state_range')['Sea_State%'].sum().reset_index()
        range_df['Sea_State_Range'] = range_profile_sea_state['sea_state_range']
        range_df['Sea_State%'] = range_profile_sea_state['Sea_State%']

        data_profile['me_time_range'] = pd.cut(data_profile['ME_Time'], bins=list(range(0, int(data_profile['ME_Time'].max()) + user_me_time_range, user_me_time_range))).astype(str)
        range_profile_me_time = data_profile.groupby('me_time_range')['ME_Time%'].sum().reset_index()
        range_df['ME_Time_Range'] = range_profile_me_time['me_time_range']
        range_df['ME_Time%'] = range_profile_me_time['ME_Time%']

        data_profile['ae_time_range'] = pd.cut(data_profile['AE_Time'], bins=list(range(0, int(data_profile['AE_Time'].max()) + user_ae_time_range, user_ae_time_range))).astype(str)
        range_profile_ae_time = data_profile.groupby('ae_time_range')['AE_Time%'].sum().reset_index()
        range_df['AE_Time_Range'] = range_profile_ae_time['ae_time_range']
        range_df['AE_Time%'] = range_profile_ae_time['AE_Time%']

        user_ae_running_range = 1
        data_profile['ae_running_range'] = pd.cut(data_profile['AE_Running'], bins=list(range(0, int(data_profile['AE_Running'].max()) + user_ae_running_range, user_ae_running_range))).astype(str)
        range_profile_ae_running = data_profile.groupby('ae_running_range')['AE_Running%'].sum().reset_index()
        range_df['AE_Running_Range'] = range_profile_ae_running['ae_running_range']
        range_df['AE_Running%'] = range_profile_ae_running['AE_Running%']

        # Prepare final profiles
        speed_draft_profile = pd3.iloc[:, :-14].dropna()
        parameters_range_profile = range_df.round(2).fillna('')
        parameters_profile = data_profile.round(2).iloc[:, :-6].fillna('')
       # parameters_profile = parameters_profile.where(pd.notna(parameters_profile), np.nan)  # Explicitly replace NaNs with np.nan


        # Add profiles to the final dictionary
        finaldict[index] = {'speed_draft_profile': speed_draft_profile,
                            'parameters_profile': parameters_profile,
                            'parameters_range_profile': parameters_range_profile}

    return finaldict


In [18]:
generate_sail_profiles(sail_cluster_data,
                  1,
                  user_draft_range = 2, 
                  user_speed_range = 4,
                  user_sea_state_range = 2,
                  user_me_time_range = 4,  
                  user_ae_time_range = 4   
                 )

Index(['index', 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0], dtype='object')


{1: {'speed_draft_profile':     Speed/Draft   5.0   6.0   7.0   8.0   9.0  10.0  11.0  12.0
  0           0.0  0.00  0.00  0.00  0.13  0.00  0.00  0.00  0.00
  1           1.0  0.00  0.00  0.00  0.00  0.00  0.00  0.13  0.00
  2           2.0  0.00  0.00  0.13  0.25  0.13  0.25  0.13  0.00
  3           3.0  0.00  0.00  0.13  0.13  0.25  0.38  0.38  0.00
  4           4.0  0.00  0.00  0.25  0.13  0.25  0.51  0.25  0.00
  5           5.0  0.00  0.13  0.13  0.64  0.25  0.25  0.64  0.00
  6           6.0  0.00  0.25  0.38  0.38  1.02  1.15  0.51  0.00
  7           7.0  0.00  0.25  0.13  0.76  1.02  2.04  0.51  0.00
  8           8.0  0.13  0.38  0.89  1.15  1.15  2.42  1.02  0.13
  9           9.0  0.00  1.02  0.76  1.65  2.80  2.16  1.27  0.38
  10         10.0  0.13  1.27  1.53  2.80  2.93  3.31  1.78  0.51
  11         11.0  0.13  1.78  1.53  1.53  2.42  2.93  1.27  0.25
  12         12.0  0.00  0.89  1.15  2.29  3.05  1.53  1.15  0.00
  13         13.0  0.13  0.76  1.40  2.29  1.65  2

In [52]:
def generate_sail_profiles(cluster_data, 
                      no_of_profiles,
                      user_draft_range=2, 
                      user_speed_range=4,
                      user_sea_state_range=2,
                      user_me_time_range=4,  
                      user_ae_time_range=4    
                     ):
    
    global finaldict
    finaldict = {}
    
    for index in range(1, no_of_profiles + 1):
      
        # Filter data for the cluster
        data = cluster_data[cluster_data['Cluster'] == index][['speed', 'mean_draft', 'me_actual_steaming_time', 'sea_state',
                                                               'ae_t_steaming', 'aux_running', 'blr_running']].copy()

        # Convert columns to numeric and round them
        data['speed'] = pd.to_numeric(data['speed'], errors='coerce').round(0)
        data['mean_draft'] = pd.to_numeric(data['mean_draft'], errors='coerce').round(0)

        # Create a pivot table for speed and draft data
        melt = pd.melt(data, id_vars=['mean_draft'], value_vars=['speed'])
        melt['speed'] = melt['value']
        pd2 = (melt.pivot_table(index='speed', columns='mean_draft', values='value', aggfunc=len, fill_value=0)
               .reset_index()
               .rename_axis(None, axis=1))
        pd2 = pd2.set_index('speed')
        pd2.index.name = None
        total = np.sum(pd2.to_numpy())
        pd2 = round((pd2 * 100 / total), 2)
        pd3 = pd2.reset_index()
        print(pd3.columns)


        # Rename the column
        pd3.rename(columns={'index': 'Speed/Draft'}, inplace=True)
        pd3['Draft'] = pd.Series(pd3.columns[1:])

        # Get draft and speed percentages
        mean_draft_percent_df = (data['mean_draft'].round(0).value_counts(normalize=True) * 100).round(1).sort_index()
        pd3['Draft_%'] = pd3['Draft'].map(dict(zip(mean_draft_percent_df.index, mean_draft_percent_df)))

        speed_percent_df = (data['speed'].round(0).value_counts(normalize=True) * 100).round(1).sort_index()
        pd3['Speed'] = pd3['Speed/Draft']
        pd3['Speed_%'] = pd3['Speed/Draft'].map(dict(zip(speed_percent_df.index, speed_percent_df)))

        # Extend the DataFrame to match the desired length (30 rows)
        pd3 = pd3.reindex(range(30))

        # Create percentage columns for Sea State, ME steaming time, AE steaming time, AE running, and BLR running
        ss_df = (data['sea_state'].round(0).value_counts(normalize=True) * 100).round(1).sort_index()
        pd3['Sea_State'] = pd.Series(sorted(abs(data['sea_state'] - 10).round().value_counts().index))
        pd3['Sea_State%'] = pd3['Sea_State'].map(dict(zip(abs(ss_df.index - 10)[::-1], ss_df)))

        me_steam_time_df = (data['me_actual_steaming_time'].round(0).value_counts(normalize=True) * 100).round(1).sort_index()
        pd3['ME_Time'] = pd.Series(sorted(data['me_actual_steaming_time'].round().value_counts().index))
        pd3['ME_Time%'] = pd3['ME_Time'].map(dict(zip(me_steam_time_df.index, me_steam_time_df)))

        ae_steam_time_df = (data['ae_t_steaming'].round(0).value_counts(normalize=True) * 100).round(1).sort_index()
        pd3['AE_Time'] = pd.Series(sorted(data['ae_t_steaming'].round().value_counts().index))
        pd3['AE_Time%'] = pd3['AE_Time'].map(dict(zip(ae_steam_time_df.index, ae_steam_time_df)))

        ae_running_df = (data['aux_running'].round(0).value_counts(normalize=True) * 100).round(1).sort_index()
        pd3['AE_Running'] = pd.Series(sorted(data['aux_running'].round().value_counts().index))
        pd3['AE_Running%'] = pd3['AE_Running'].map(dict(zip(ae_running_df.index, ae_running_df)))

        blr_running_df = (data['blr_running'].round(0).value_counts(normalize=True) * 100).round(1).sort_index()
        pd3['BLR_Running'] = pd.Series(sorted(data['blr_running'].round().value_counts().index))
        pd3['BLR_Running%'] = pd3['BLR_Running'].map(dict(zip(blr_running_df.index, blr_running_df)))

        # Create range DataFrame
        range_df = pd.DataFrame(index=range(10))

        # Process ranges for draft, speed, sea state, ME time, AE time, AE running, and BLR running
        data_profile = pd3.iloc[:, -14:].copy()

        data_profile['draft_range'] = pd.cut(data_profile['Draft'], bins=list(range(0, int(data_profile['Draft'].max()) + user_draft_range, user_draft_range))).astype(str)
        range_profile_draft = data_profile.groupby('draft_range')['Draft_%'].sum().reset_index()
        range_df['Draft_Range'] = range_profile_draft['draft_range']
        range_df['Draft_%'] = range_profile_draft['Draft_%']

        data_profile['speed_range'] = pd.cut(data_profile['Speed'], bins=list(range(0, int(data_profile['Speed'].max()) + user_speed_range, user_speed_range))).astype(str)
        range_profile_speed = data_profile.groupby('speed_range')['Speed_%'].sum().reset_index()
        range_df['Speed_Range'] = range_profile_speed['speed_range']
        range_df['Speed_%'] = range_profile_speed['Speed_%']

        data_profile['sea_state_range'] = pd.cut(data_profile['Sea_State'], bins=list(range(0, int(data_profile['Sea_State'].max()) + user_sea_state_range, user_sea_state_range))).astype(str)
        range_profile_sea_state = data_profile.groupby('sea_state_range')['Sea_State%'].sum().reset_index()
        range_df['Sea_State_Range'] = range_profile_sea_state['sea_state_range']
        range_df['Sea_State%'] = range_profile_sea_state['Sea_State%']

        data_profile['me_time_range'] = pd.cut(data_profile['ME_Time'], bins=list(range(0, int(data_profile['ME_Time'].max()) + user_me_time_range, user_me_time_range))).astype(str)
        range_profile_me_time = data_profile.groupby('me_time_range')['ME_Time%'].sum().reset_index()
        range_df['ME_Time_Range'] = range_profile_me_time['me_time_range']
        range_df['ME_Time%'] = range_profile_me_time['ME_Time%']

        data_profile['ae_time_range'] = pd.cut(data_profile['AE_Time'], bins=list(range(0, int(data_profile['AE_Time'].max()) + user_ae_time_range, user_ae_time_range))).astype(str)
        range_profile_ae_time = data_profile.groupby('ae_time_range')['AE_Time%'].sum().reset_index()
        range_df['AE_Time_Range'] = range_profile_ae_time['ae_time_range']
        range_df['AE_Time%'] = range_profile_ae_time['AE_Time%']

        user_ae_running_range = 1
        data_profile['ae_running_range'] = pd.cut(data_profile['AE_Running'], bins=list(range(0, int(data_profile['AE_Running'].max()) + user_ae_running_range, user_ae_running_range))).astype(str)
        range_profile_ae_running = data_profile.groupby('ae_running_range')['AE_Running%'].sum().reset_index()
        range_df['AE_Running_Range'] = range_profile_ae_running['ae_running_range']
        range_df['AE_Running%'] = range_profile_ae_running['AE_Running%']

        # Prepare final profiles
        speed_draft_profile = pd3.iloc[:, :-14].dropna()
        parameters_range_profile = range_df.round(2).fillna('')
        parameters_profile = data_profile.round(2).iloc[:, :-6].fillna('')
       # parameters_profile = parameters_profile.where(pd.notna(parameters_profile), np.nan)  # Explicitly replace NaNs with np.nan


        # Add profiles to the final dictionary
        finaldict[index] = {'speed_draft_profile': speed_draft_profile,
                            'parameters_profile': parameters_profile,
                            'parameters_range_profile': parameters_range_profile}

    return finaldict


In [54]:
generate_sail_profiles(sail_cluster_data,
                  1,
                  user_draft_range = 2, 
                  user_speed_range = 4,
                  user_sea_state_range = 2,
                  user_me_time_range = 4,  
                  user_ae_time_range = 4   
                 )

Index(['index', 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0], dtype='object')


{1: {'speed_draft_profile':     Speed/Draft   5.0   6.0   7.0   8.0   9.0  10.0  11.0  12.0
  0           0.0  0.00  0.06  0.00  0.06  0.00  0.00  0.00  0.00
  1           1.0  0.00  0.06  0.11  0.06  0.00  0.00  0.00  0.06
  2           2.0  0.00  0.00  0.06  0.28  0.17  0.28  0.34  0.06
  3           3.0  0.00  0.06  0.11  0.40  0.40  0.57  0.51  0.06
  4           4.0  0.06  0.17  0.40  0.63  0.63  0.68  0.57  0.11
  5           5.0  0.00  0.17  0.34  1.02  0.40  1.31  1.02  0.00
  6           6.0  0.11  0.34  0.63  0.91  1.36  1.93  0.80  0.11
  7           7.0  0.00  0.28  0.68  1.31  1.71  1.99  0.63  0.23
  8           8.0  0.11  0.23  1.02  1.88  2.33  2.79  1.71  0.34
  9           9.0  0.00  0.11  1.19  2.44  3.24  3.13  1.53  0.57
  10         10.0  0.11  0.51  1.93  3.58  3.35  3.47  1.99  0.45
  11         11.0  0.11  0.51  0.97  2.62  3.24  2.96  1.02  0.11
  12         12.0  0.00  0.28  1.31  3.47  1.65  2.79  0.97  0.40
  13         13.0  0.00  0.23  0.97  1.48  1.36  1

# without function

In [ ]:
import pandas as pd
import numpy as np



no_of_profiles = 3  # Example value



for index in range(1, no_of_profiles + 1):
    data = sail_cluster_data[sail_cluster_data['Cluster'] == index][
        ['speed', 'mean_draft', 'me_actual_steaming_time', 'sea_state',
         'ae_t_steaming', 'aux_running', 'blr_running']
    ].copy()

data['speed'] = data['speed'].apply(pd.to_numeric, args=('coerce',)).round(0)
data['mean_draft'] = data['mean_draft'].apply(pd.to_numeric, args=('coerce',)).round(0)

melt = pd.melt(data, id_vars=['mean_draft'], value_vars=['speed'])
melt['speed'] = melt['value']
pd2 = melt.pivot_table(index='speed', columns='mean_draft', values='value', aggfunc=np.size, fill_value=0).reset_index().rename_axis(None, axis=1)
pd2 = pd2.set_index('speed')
pd2.index.name = None
total = np.sum(pd2.to_numpy())
pd2 = round((pd2 * 100 / total), 2)

pd3 = pd2.reset_index()
pd3.rename(columns={'index': 'Speed/Draft'}, inplace=True)
pd3['Draft'] = pd.Series(list(pd3.columns[1:]))
print(pd3)


In [ ]:
mean_draft_percent_df = pd.DataFrame((data['mean_draft'].round(0).value_counts(normalize=True) * 100).round(1).sort_index())

# Create a dictionary to map draft values to their proportions
draft_percent_dict = dict(zip(mean_draft_percent_df.index, mean_draft_percent_df['proportion']))

# Map the Draft column in pd3 to the corresponding proportions
pd3['Draft_%'] = pd3['Draft'].map(draft_percent_dict)


In [ ]:
pd3

In [ ]:

speed_percent_df = pd.DataFrame((data['speed'].round(0).value_counts(normalize=True) * 100).round(1).sort_index())
speed_percent_df.columns = ['proportion']  # Rename the column to match your data

# Create a dictionary to map speed values to their proportions
speed_percent_dict = dict(zip(speed_percent_df.index, speed_percent_df['proportion']))

# Map the Speed column in pd3 to the corresponding proportions
pd3['Speed_%'] = pd3['Speed/Draft'].map(speed_percent_dict)


In [1]:
parameters_range_profile = pd.DataFrame({
    'Draft_Range': range_profile_draft['draft_range'].astype(str),
    'Draft_%': range_profile_draft['Draft_%'],
    'Speed_Range': range_profile_speed['speed_range'].astype(str),
    'Speed_%': range_profile_speed['Speed_%'],
    'Sea_State_Range': range_profile_sea_state['sea_state_range'].astype(str),
    'Sea_State%': range_profile_sea_state['Sea_State%'],
    'AE_Time_Range': range_profile_ae_time['ae_time_range'].astype(str),
    'AE_Time%': range_profile_ae_time['AE_Time%'],
    'AE_Running_Range': range_profile_ae_running['ae_running_range'].astype(str),
    'AE_Running%': range_profile_ae_running['AE_Running%'],
    'BLR_Running': data_profile['BLR_Running'].astype(str),
    'BLR_Running%': data_profile['BLR_Running%']
})

NameError: name 'pd' is not defined

In [ ]:
finaldict

In [ ]:
mean_draft_percent_df = pd.DataFrame((data['mean_draft'].round(0).value_counts(normalize=True) * 100).round(1).sort_index())
pd3['Draft_%'] = pd3['Draft'].map(dict(zip(list(mean_draft_percent_df.index), mean_draft_percent_df['mean_draft'])))


In [ ]:
range_df['Draft_Range'] = pd.Series(range_profile_draft['draft_range']).astype(str)
range_df

In [ ]:
# Function to extract numeric values from draft range
def extract_numbers(draft_range):
    try:
        if pd.isna(draft_range):
            return (np.nan, np.nan)
        lower, upper = draft_range.strip('()[]').split(', ')
        return (int(lower), int(upper))
    except Exception as e:
        print(f"Error parsing {draft_range}: {e}")
        return (np.nan, np.nan)

# Filter out nan values for sorting and create a sorted list of valid ranges
valid_ranges = range_df['Draft_Range'].dropna().unique()
valid_ranges_sorted = sorted(valid_ranges, key=lambda x: extract_numbers(x))

# Convert 'Draft_Range' to categorical type with the ordered categories
range_df['Draft_Range'] = pd.Categorical(
    range_df['Draft_Range'],
    categories=valid_ranges_sorted,
    ordered=True
)

# Sort the DataFrame by the 'Draft_Range'
range_df.sort_values(by='Draft_Range', inplace=True)

# Optionally, reset index for clean DataFrame
range_df.reset_index(drop=True, inplace=True)

print(range_df)

In [ ]:
mean_draft_percent_df

In [ ]:
pd3['Draft_%']

# without global 

In [ ]:
import pandas as pd
import numpy as np

def generate_sail_profiles(sail_cluster_data, no_of_profiles,
                            user_draft_range=2, 
                            user_speed_range=4,
                            user_sea_state_range=2,
                            user_me_time_range=4,  
                            user_ae_time_range=4):
    
    finaldict = {}
    
    for index in range(1, no_of_profiles + 1):
        data = sail_cluster_data[sail_cluster_data['Cluster'] == index][
            ['speed', 'mean_draft', 'me_actual_steaming_time', 'sea_state',
             'ae_t_steaming', 'aux_running', 'blr_running']
        ].copy()

        data['speed'] = data['speed'].apply(pd.to_numeric, errors='coerce').round(0)
        data['mean_draft'] = data['mean_draft'].apply(pd.to_numeric, errors='coerce').round(0)

        melt = pd.melt(data, id_vars=['mean_draft'], value_vars=['speed'])
        melt['speed'] = melt['value']
        pd2 = melt.pivot_table(index='speed', columns='mean_draft', values='value', aggfunc=np.size, fill_value=0).reset_index().rename_axis(None, axis=1)
        pd2 = pd2.set_index('speed')
        pd2.index.name = None
        total = np.sum(pd2.to_numpy())
        pd2 = round((pd2 * 100 / total), 2)

        pd3 = pd2.reset_index()
        pd3.rename(columns={'index': 'Speed/Draft'}, inplace=True)
        pd3['Draft'] = pd.Series(list(pd3.columns[1:]))
        
        # Value count
        mean_draft_percent_df = pd.DataFrame((data['mean_draft'].round(0).value_counts(normalize=True) * 100).round(1).sort_index())
        pd3['Draft_%'] = pd3['Draft'].map(dict(zip(list(mean_draft_percent_df.index), mean_draft_percent_df['mean_draft'])))

        speed_percent_df = pd.DataFrame((data['speed'].round(0).value_counts(normalize=True) * 100).round(1).sort_index())
        pd3['Speed'] = pd3['Speed/Draft']
        pd3['Speed_%'] = pd3['Speed/Draft'].map(dict(zip(list(speed_percent_df.index), speed_percent_df['speed'])))

        # Extend the DataFrame to match the series length
        pd3 = pd3.reindex(range(30))

        # Sea State
        ss_df = pd.DataFrame((data['sea_state'].round(0).value_counts(normalize=True) * 100).round(1).sort_index())
        pd3['Sea_State'] = pd.Series(sorted(list(abs(data['sea_state'] - 10).apply(lambda x: round(x)).value_counts().index)))
        pd3['Sea_State%'] = pd3['Sea_State'].map(dict(zip(list(abs(ss_df.index - 10))[::-1], ss_df['sea_state'])))

        # ME Steaming Time
        me_steam_time_df = pd.DataFrame((data['me_actual_steaming_time'].round(0).value_counts(normalize=True) * 100).round(1).sort_index())
        pd3['ME_Time'] = pd.Series(sorted(list(data['me_actual_steaming_time'].apply(lambda x: round(x)).value_counts().index)))
        pd3['ME_Time%'] = pd3['ME_Time'].map(dict(zip(list(me_steam_time_df.index), me_steam_time_df['me_actual_steaming_time'])))

        # AE Steaming Time
        ae_steam_time_df = pd.DataFrame((data['ae_t_steaming'].round(0).value_counts(normalize=True) * 100).round(1).sort_index())
        pd3['AE_Time'] = pd.Series(sorted(list(data['ae_t_steaming'].apply(lambda x: round(x)).value_counts().index)))
        pd3['AE_Time%'] = pd3['AE_Time'].map(dict(zip(list(ae_steam_time_df.index), ae_steam_time_df['ae_t_steaming'])))

        # AE Running
        ae_running_df = pd.DataFrame((data['aux_running'].round(0).value_counts(normalize=True) * 100).round(1).sort_index())
        pd3['AE_Running'] = pd.Series(sorted(list(data['aux_running'].apply(lambda x: round(x)).value_counts().index)))
        pd3['AE_Running%'] = pd3['AE_Running'].map(dict(zip(list(ae_running_df.index), ae_running_df['aux_running'])))

        # BLR Running
        blr_running_df = pd.DataFrame((data['blr_running'].round(0).value_counts(normalize=True) * 100).round(1).sort_index())
        pd3['BLR_Running'] = pd.Series(sorted(list(data['blr_running'].apply(lambda x: round(x)).value_counts().index)))
        pd3['BLR_Running%'] = pd3['BLR_Running'].map(dict(zip(list(blr_running_df.index), blr_running_df['blr_running'])))

        range_df = pd.DataFrame(index=range(10))

        data_profile = pd3.iloc[:, -14:]

        # Draft Range
        data_profile['draft_range'] = pd.cut(x=data_profile['Draft'], bins=list(range(0, int(data_profile['Draft'].max()) + user_draft_range, user_draft_range))).astype(str)
        range_profile_draft = data_profile.bfill().groupby('draft_range')['Draft_%'].sum().reset_index()
        range_df['Draft_Range'] = pd.Series(range_profile_draft['draft_range']).astype(str)
        range_df['Draft_%'] = pd.Series(range_profile_draft['Draft_%'])

        # Speed Range
        data_profile['speed_range'] = pd.cut(x=data_profile['Speed'], bins=list(range(0, int(data_profile['Speed'].max()) + user_speed_range, user_speed_range))).astype(str)
        range_profile_speed = data_profile.bfill().groupby('speed_range')['Speed_%'].sum().reset_index()
        range_df['Speed_Range'] = pd.Series(range_profile_speed['speed_range']).astype(str)
        range_df['Speed_%'] = pd.Series(range_profile_speed['Speed_%'])

        # Sea State Range
        data_profile['sea_state_range'] = pd.cut(x=data_profile['Sea_State'], bins=list(range(0, int(data_profile['Sea_State'].max()) + user_sea_state_range, user_sea_state_range))).astype(str)
        range_profile_sea_state = data_profile.bfill().groupby('sea_state_range')['Sea_State%'].sum().reset_index()
        range_df['Sea_State_Range'] = pd.Series(range_profile_sea_state['sea_state_range']).astype(str)
        range_df['Sea_State%'] = pd.Series(range_profile_sea_state['Sea_State%'])

        # ME Time Range
        data_profile['me_time_range'] = pd.cut(x=data_profile['ME_Time'], bins=list(range(0, int(data_profile['ME_Time'].max()) + user_me_time_range, user_me_time_range))).astype(str)
        range_profile_me_time = data_profile.bfill().groupby('me_time_range')['ME_Time%'].sum().reset_index()
        range_df['ME_Time_Range'] = pd.Series(range_profile_me_time['me_time_range']).astype(str)
        range_df['ME_Time%'] = pd.Series(range_profile_me_time['ME_Time%'])

        # AE Time Range
        data_profile['ae_time_range'] = pd.cut(x=data_profile['AE_Time'], bins=list(range(0, int(data_profile['AE_Time'].max()) + user_ae_time_range, user_ae_time_range))).astype(str)
        range_profile_ae_time = data_profile.bfill().groupby('ae_time_range')['AE_Time%'].sum().reset_index()
        range_df['AE_Time_Range'] = pd.Series(range_profile_ae_time['ae_time_range']).astype(str)
        range_df['AE_Time%'] = pd.Series(range_profile_ae_time['AE_Time%'])

        # AE Running Range
        data_profile['ae_running_range'] = pd.cut(x=data_profile['AE_Running'], bins=list(range(0, int(data_profile['AE_Running'].max()) + 1, 1))).astype(str)
        range_profile_ae_running = data_profile.bfill().groupby('ae_running_range')['AE_Running%'].sum().reset_index()
        range_df['AE_Running_Range'] = pd.Series(range_profile_ae_running['ae_running_range']).astype(str)
        range_df['AE_Running%'] = pd.Series(range_profile_ae_running['AE_Running%'])


        # BLR Running Range
#         data_profile['blr_running_range'] = pd.cut(x=data_profile['BLR_Running'], bins=list(range(0, int(data_profile['AE_Running'].max()) + 1, 1))).astype(str)
#         range_profile_blr_running = data_profile.bfill().groupby('blr_running_range')['BLR_Running%'].sum().reset_index()
#         range_df['BLR_Running_Range'] = pd.Series(data_profile['blr_running_range'].values).astype(str)
#         range_df['BLR_Running%'] = pd.Series(data_profile['BLR_Running%'])

        range_df['BLR_Running'] = pd.Series(data_profile['BLR_Running'].values).astype(str)
        range_df['BLR_Running%'] = pd.Series(data_profile['BLR_Running%'])

        finaldict = {}

        # Store the profiles in finaldict for each index
        finaldict[f'speed_draft_profile_{index}'] = speed_draft_profile.to_dict(orient='records')
        finaldict[f'parameters_range_profile_{index}'] = parameters_range_profile.to_dict(orient='records')
        finaldict[f'parameters_profile_{index}'] = parameters_profile.to_dict(orient='records')



        # Return the final dictionary
        return finaldict

In [ ]:
parameters_profile

In [ ]:
speed_draft_profile

In [ ]:
parameters_range_profile

In [ ]:
generate_sail_profiles(sail_cluster_data,
                  1,
                  user_draft_range = 2, 
                  user_speed_range = 4,
                  user_sea_state_range = 2,
                  user_me_time_range = 4,  
                  user_ae_time_range = 4   
                 )

In [ ]:
finaldict.keys()

# Find Subset Data Profile

In [ ]:
subset_data_sail = pd.read_excel("Subset_SAIL_DF.xlsx" )
subset_data_sail

In [ ]:
generate_sail_profiles(subset_data_sail,
                  1,
                  user_draft_range = 2, 
                  user_speed_range = 4,
                  user_sea_state_range = 2,
                  user_me_time_range = 4,  
                  user_ae_time_range = 4   
                 )

In [ ]:
 finaldict.keys()

In [ ]:
import pandas as pd

for i in finaldict.keys():
    # Create a Pandas Excel writer using XlsxWriter as the engine
    with pd.ExcelWriter(f"Subset_Sail_{i}.xlsx", engine='xlsxwriter') as writer:
        # Write each DataFrame to a different sheet
        finaldict[i]['speed_draft_profile'].to_excel(writer, sheet_name='Speed_Draft_Profile', index=False)
        finaldict[i]['parameters_profile'].to_excel(writer, sheet_name='Parameters_Profile', index=False)
        finaldict[i]['parameters_range_profile'].to_excel(writer, sheet_name='Parameters_Range_Profile', index=False)


In [ ]:
cluster_data